<a href="https://colab.research.google.com/github/BerkKilincer-dev/SoftIto/blob/main/transformers-llm/06_kv_cache_ve_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Çıkarımı ve İnce Ayar: KV Cache ve LoRA

**Bölüm A – KV Cache:** Otoregresif üretimde her yeni token için geçmişin Key/Value vektörlerini baştan hesaplamak yerine **önbellekte** tutarak üretimi hızlandırıyoruz. Önbelleksiz ve önbellekli üretimi aynı model üzerinde karşılaştırıyoruz.

**Bölüm B – LoRA:** Milyarlarca parametreli bir modelin tamamını eğitmek yerine, ağırlık güncellemesini iki küçük matrisin çarpımı olarak öğreniyoruz. LoRA'yı sıfırdan yazıp tam ince ayarla (full fine-tuning) karşılaştırıyoruz.

In [ ]:
import math
import time
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
plt.rcParams["figure.dpi"] = 100
cihaz = "cuda" if torch.cuda.is_available() else "cpu"
print("Cihaz:", cihaz)

---
# Bölüm A — KV Cache

### Problem

Bir decoder modeli metni **token token** üretir. $t$. tokenı üretirken $t$ pozisyonundaki sorgu $q_t$, önceki **tüm** pozisyonların anahtar ve değerlerine ($k_1 \dots k_t$, $v_1 \dots v_t$) bakar.

Önbelleksiz üretimde her adımda **tüm dizi** modelden tekrar geçirilir:
- Adım 1: 1 token işle → Adım 2: 2 token işle → … → Adım $n$: $n$ token işle
- Toplam: $1 + 2 + \dots + n = O(n^2)$ token işleme

Oysa causal maske sayesinde **geçmiş tokenların K ve V değerleri hiç değişmez**. Onları bir kez hesaplayıp saklarsak her adımda yalnızca **yeni tokenı** işlememiz yeter → toplam $O(n)$.

```
Önbelleksiz:  [Ben] → [Ben bugün] → [Ben bugün okula] → [Ben bugün okula gittim]   (her seferinde hepsi)
KV Cache   :  [Ben] → [bugün]     → [okula]           → [gittim]                   (sadece yeni token)
               K,V önbelleği:  [k₁ v₁] → [k₁ v₁ k₂ v₂] → ...  (her adımda bir satır eklenir)
```

In [ ]:
class Dikkat(nn.Module):
    def __init__(self, d, n_kafa):
        super().__init__()
        self.n_kafa, self.d_kafa = n_kafa, d // n_kafa
        self.qkv = nn.Linear(d, 3 * d)
        self.o = nn.Linear(d, d)

    def forward(self, x, onbellek=None):
        B, T, D = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.n_kafa, self.d_kafa).permute(2, 0, 3, 1, 4)

        if onbellek is not None:
            if "k" in onbellek:                                  # geçmişin K,V'lerini başa ekle
                k = torch.cat([onbellek["k"], k], dim=2)
                v = torch.cat([onbellek["v"], v], dim=2)
            onbellek["k"], onbellek["v"] = k, v                 # güncellenmiş önbelleği sakla

        T_q, T_k = q.size(2), k.size(2)
        # Sorgu i'nin mutlak konumu (T_k - T_q + i); kendisine ve öncesine bakabilir
        maske = torch.ones(T_q, T_k, dtype=torch.bool, device=x.device).tril(diagonal=T_k - T_q)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=maske)
        return self.o(y.transpose(1, 2).reshape(B, T, D))


class Blok(nn.Module):
    def __init__(self, d, n_kafa):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.dikkat = Dikkat(d, n_kafa)
        self.ffn = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x, onbellek=None):
        x = x + self.dikkat(self.ln1(x), onbellek)
        return x + self.ffn(self.ln2(x))


class MiniGPT(nn.Module):
    def __init__(self, sozluk=1000, d=256, n_katman=6, n_kafa=8, maks_uzunluk=1024):
        super().__init__()
        self.tok = nn.Embedding(sozluk, d)
        self.poz = nn.Embedding(maks_uzunluk, d)
        self.bloklar = nn.ModuleList([Blok(d, n_kafa) for _ in range(n_katman)])
        self.ln = nn.LayerNorm(d)
        self.bas = nn.Linear(d, sozluk, bias=False)

    def forward(self, idx, onbellekler=None, baslangic=0):
        T = idx.size(1)
        x = self.tok(idx) + self.poz(torch.arange(baslangic, baslangic + T, device=idx.device))
        for i, blok in enumerate(self.bloklar):
            x = blok(x, None if onbellekler is None else onbellekler[i])
        return self.bas(self.ln(x))

    @torch.no_grad()
    def uret_onbelleksiz(self, idx, n_yeni):
        for _ in range(n_yeni):
            logit = self(idx)                                    # TÜM dizi her seferinde yeniden işlenir
            idx = torch.cat([idx, logit[:, -1].argmax(-1, keepdim=True)], dim=1)
        return idx

    @torch.no_grad()
    def uret_kv_cache(self, idx, n_yeni):
        onbellekler = [{} for _ in self.bloklar]
        logit = self(idx, onbellekler)                           # 1) prefill: istemi bir kez işle
        konum = idx.size(1)
        for _ in range(n_yeni):
            sonraki = logit[:, -1].argmax(-1, keepdim=True)
            idx = torch.cat([idx, sonraki], dim=1)
            logit = self(sonraki, onbellekler, baslangic=konum)  # 2) decode: sadece YENİ token
            konum += 1
        return idx

model = MiniGPT().to(cihaz).eval()
print(f"Model: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M parametre")

### Doğruluk: iki yöntem birebir aynı çıktıyı üretmeli

In [ ]:
istem = torch.randint(0, 1000, (1, 16), device=cihaz)
a = model.uret_onbelleksiz(istem, 50)
b = model.uret_kv_cache(istem, 50)
print("Üretilen tokenlar aynı mı? →", torch.equal(a, b))
print("İlk 20 token:", b[0, 16:36].tolist())

# Logit seviyesinde kontrol: diziyi tek seferde işlemek ile token token önbellekle işlemek
with torch.no_grad():
    tek_seferde = model(b)
    onbellekler = [{} for _ in model.bloklar]
    adim_adim = torch.cat([model(b[:, t:t + 1], onbellekler, baslangic=t) for t in range(b.size(1))], dim=1)
print(f"Logit max farkı: {(tek_seferde - adim_adim).abs().max():.2e}  (sadece kayan nokta yuvarlama farkı)")

### Hız karşılaştırması

In [ ]:
def sure_olc(fonk, *args):
    if cihaz == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    fonk(*args)
    if cihaz == "cuda":
        torch.cuda.synchronize()
    return time.perf_counter() - t0

satirlar = []
for n in [32, 64, 128, 256, 512]:
    t_yok = sure_olc(model.uret_onbelleksiz, istem, n)
    t_var = sure_olc(model.uret_kv_cache, istem, n)
    satirlar.append({"üretilen token": n, "önbelleksiz (sn)": t_yok, "KV cache (sn)": t_var,
                     "hızlanma": t_yok / t_var})
hiz = pd.DataFrame(satirlar)

plt.figure(figsize=(9, 4.5))
plt.plot(hiz["üretilen token"], hiz["önbelleksiz (sn)"], "o-", label="önbelleksiz  ~ O(n²)")
plt.plot(hiz["üretilen token"], hiz["KV cache (sn)"], "s-", label="KV cache  ~ O(n)")
plt.xlabel("üretilen token sayısı"); plt.ylabel("toplam süre (sn)")
plt.title("Üretim süresi"); plt.legend(); plt.grid(alpha=0.3); plt.show()
hiz.round(3)

### Bedel: bellek

KV cache hızı **bellekle** satın alır. Token başına önbellek boyutu:

$$\text{bayt/token} = 2_{(K,V)} \times n_{\text{katman}} \times n_{\text{KV kafa}} \times d_{\text{kafa}} \times \text{bayt/eleman}$$

Uzun bağlamlarda ve çok kullanıcılı sunucularda KV cache, model ağırlıklarından bile **daha fazla** bellek tutabilir. Bu yüzden modern modeller **GQA** (Grouped-Query Attention: birkaç sorgu kafası bir K/V kafasını paylaşır) veya **MQA** (tek K/V kafası) kullanır; vLLM gibi sunucular **PagedAttention** ile önbelleği sayfalar halinde yönetir.

In [ ]:
modeller = {
    #               katman, sorgu kafası, KV kafası, d_kafa
    "LLaMA-2-7B (MHA)":  (32, 32, 32, 128),
    "LLaMA-3-8B (GQA)":  (32, 32, 8, 128),
    "LLaMA-3-70B (GQA)": (80, 64, 8, 128),
    "Mistral-7B (GQA)":  (32, 32, 8, 128),
}
satirlar = []
for isim, (L, h_q, h_kv, d_h) in modeller.items():
    bayt_token = 2 * L * h_kv * d_h * 2                       # FP16
    satirlar.append({"Model": isim,
                     "KB / token": bayt_token / 1024,
                     "4K bağlam (GB)": bayt_token * 4096 / 2**30,
                     "32K bağlam (GB)": bayt_token * 32_768 / 2**30,
                     "128K bağlam (GB)": bayt_token * 131_072 / 2**30})
pd.DataFrame(satirlar).set_index("Model").round(2)

---
# Bölüm B — LoRA: Low-Rank Adaptation (Hu vd., 2021)

### Fikir

Tam ince ayarda her ağırlık matrisi $W \in \mathbb{R}^{d \times k}$ güncellenir: $W' = W + \Delta W$. LoRA'nın gözlemi: ince ayar sırasındaki güncelleme $\Delta W$'nin **rankı düşüktür**. O halde onu iki küçük matrisin çarpımı olarak öğrenebiliriz:

$$W' = W_0 + \frac{\alpha}{r}\, B A, \qquad B \in \mathbb{R}^{d \times r},\; A \in \mathbb{R}^{r \times k},\; r \ll \min(d, k)$$

- $W_0$ **dondurulur** (gradyan yok, optimizer durumu yok).
- $A$ rastgele, $B$ **sıfır** başlatılır → eğitim başında $\Delta W = 0$, model orijinaliyle birebir aynı başlar.
- $d = k = 4096$, $r = 8$ için: tam güncelleme 16.7 M parametre, LoRA ise $8 \times (4096 + 4096) = 65$ K parametre → **%0.4**.
- Eğitimden sonra $BA$ ağırlığa **birleştirilebilir** (merge) → çıkarımda ek gecikme sıfır.
- Bir temel model + görev başına birkaç MB'lık adaptör dosyası → aynı GPU'da yüzlerce görev.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, temel: nn.Linear, r=4, alfa=8):
        super().__init__()
        self.temel = temel
        for p in self.temel.parameters():
            p.requires_grad = False                                      # orijinal ağırlıklar donduruldu
        self.r, self.olcek = r, alfa / r
        cihaz = temel.weight.device
        self.A = nn.Parameter(torch.randn(r, temel.in_features, device=cihaz) / math.sqrt(temel.in_features))
        self.B = nn.Parameter(torch.zeros(temel.out_features, r, device=cihaz))   # sıfır → başlangıçta ΔW = 0
        self.aktif = True

    def forward(self, x):
        y = self.temel(x)
        if self.aktif:
            y = y + (x @ self.A.T @ self.B.T) * self.olcek
        return y

    @torch.no_grad()
    def birlestir(self):
        """W ← W + (α/r)·B·A ; adaptörü ağırlığa gömer, çıkarımda ek maliyet kalmaz."""
        birlesik = nn.Linear(self.temel.in_features, self.temel.out_features).to(self.A.device)
        birlesik.weight.copy_(self.temel.weight + self.olcek * self.B @ self.A)
        birlesik.bias.copy_(self.temel.bias)
        return birlesik


def lora_ekle(model, r=4, alfa=8):
    """Modeldeki tüm nn.Linear katmanlarını LoRALinear ile sarar."""
    for isim, alt in model.named_children():
        if isinstance(alt, nn.Linear):
            setattr(model, isim, LoRALinear(alt, r, alfa))
        else:
            lora_ekle(alt, r, alfa)
    return model


def parametre_ozeti(model):
    egitilebilir = sum(p.numel() for p in model.parameters() if p.requires_grad)
    toplam = sum(p.numel() for p in model.parameters())
    return egitilebilir, toplam

### Deney kurulumu

1. **Ön eğitim (pretraining):** Bir MLP'yi el yazısı rakamlar (sklearn `digits`) üzerinde eğitiyoruz.
2. **Yeni görev (domain shift):** Aynı rakamlar ama **renkleri ters çevrilmiş** ve **yatay aynalanmış**. Ön eğitimli model burada çuvallıyor.
3. Yeni göreve uyum: **tam ince ayar** vs **LoRA** (r = 4).

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

rakamlar = load_digits()
X = torch.tensor(rakamlar.data / 16.0, dtype=torch.float32)
y = torch.tensor(rakamlar.target)
X_egt, X_tst, y_egt, y_tst = [t.to(cihaz) for t in train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)]

def yeni_gorev(X):
    goruntu = 1.0 - X.view(-1, 8, 8)              # renkleri ters çevir
    return goruntu.flip(-1).reshape(-1, 64)        # yatay ayna

Xb_egt, Xb_tst = yeni_gorev(X_egt), yeni_gorev(X_tst)

fig, eksenler = plt.subplots(2, 8, figsize=(13, 3.5))
for i in range(8):
    eksenler[0, i].imshow(X_egt[i].view(8, 8).cpu(), cmap="gray_r"); eksenler[0, i].axis("off")
    eksenler[1, i].imshow(Xb_egt[i].view(8, 8).cpu(), cmap="gray_r"); eksenler[1, i].axis("off")
    eksenler[0, i].set_title(int(y_egt[i]))
eksenler[0, 0].text(-12, 4, "Görev A", fontsize=11); eksenler[1, 0].text(-12, 4, "Görev B", fontsize=11)
plt.suptitle("Üst: ön eğitim verisi | Alt: yeni görev (ters + ayna)"); plt.show()

In [ ]:
def mlp():
    return nn.Sequential(nn.Linear(64, 512), nn.ReLU(), nn.Linear(512, 512), nn.ReLU(),
                         nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, 10)).to(cihaz)

def dogruluk(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(-1) == y).float().mean().item()

def egit(model, X, y, epok=150, lr=1e-3):
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    for _ in range(epok):
        model.train()
        perm = torch.randperm(len(X), device=X.device)
        for i in range(0, len(X), 64):
            idx = perm[i:i + 64]
            kayip = F.cross_entropy(model(X[idx]), y[idx])
            opt.zero_grad(); kayip.backward(); opt.step()
    return model

torch.manual_seed(0)
temel_model = egit(mlp(), X_egt, y_egt, epok=60)
print(f"Ön eğitimli model → Görev A doğruluğu: %{dogruluk(temel_model, X_tst, y_tst) * 100:.1f}")
print(f"Ön eğitimli model → Görev B doğruluğu: %{dogruluk(temel_model, Xb_tst, y_tst) * 100:.1f}  ← yeni görevde başarısız")

In [ ]:
sonuclar = []

# 1) Tam ince ayar: tüm parametreler eğitiliyor
tam = egit(copy.deepcopy(temel_model), Xb_egt, y_egt, epok=40)
e, t = parametre_ozeti(tam)
sonuclar.append({"Yöntem": "Tam ince ayar", "Eğitilen parametre": e, "Toplam": t,
                 "Görev B": dogruluk(tam, Xb_tst, y_tst), "Görev A (unutma?)": dogruluk(tam, X_tst, y_tst)})

# 2) LoRA: farklı rank değerleriyle
lora_modelleri = {}
for r in [1, 4, 16]:
    torch.manual_seed(0)
    lora = lora_ekle(copy.deepcopy(temel_model), r=r, alfa=2 * r)
    lora = egit(lora, Xb_egt, y_egt, epok=40, lr=3e-3)
    lora_modelleri[r] = lora
    e, t = parametre_ozeti(lora)
    sonuclar.append({"Yöntem": f"LoRA r={r}", "Eğitilen parametre": e, "Toplam": t,
                     "Görev B": dogruluk(lora, Xb_tst, y_tst), "Görev A (unutma?)": dogruluk(lora, X_tst, y_tst)})

tablo = pd.DataFrame(sonuclar)
tablo["Eğitilen oran"] = (tablo["Eğitilen parametre"] / tablo["Toplam"] * 100).round(2).astype(str) + " %"
tablo[["Görev B", "Görev A (unutma?)"]] = (tablo[["Görev B", "Görev A (unutma?)"]] * 100).round(1)
tablo

### Adaptörü kapat / aç ve birleştir

LoRA'da temel ağırlıklar hiç değişmediği için adaptörü **kapatınca** orijinal model geri gelir. Tam ince ayarda ise orijinal görev bilgisi kısmen ya da tamamen **unutulabilir** (catastrophic forgetting). Tek bir temel model saklayıp görev başına küçük adaptörler takıp çıkarabiliriz.

In [ ]:
lora = lora_modelleri[4]

def adaptor_ayarla(model, aktif):
    for m in model.modules():
        if isinstance(m, LoRALinear):
            m.aktif = aktif

adaptor_ayarla(lora, False)
print(f"Adaptör KAPALI → Görev A: %{dogruluk(lora, X_tst, y_tst) * 100:.1f} | Görev B: %{dogruluk(lora, Xb_tst, y_tst) * 100:.1f}")
adaptor_ayarla(lora, True)
print(f"Adaptör AÇIK   → Görev A: %{dogruluk(lora, X_tst, y_tst) * 100:.1f} | Görev B: %{dogruluk(lora, Xb_tst, y_tst) * 100:.1f}")

# Birleştirme: LoRALinear katmanlarını düz nn.Linear'a çevir
birlesik = copy.deepcopy(lora)
for i, katman in enumerate(birlesik):
    if isinstance(katman, LoRALinear):
        birlesik[i] = katman.birlestir()
with torch.no_grad():
    fark = (lora(Xb_tst) - birlesik(Xb_tst)).abs().max().item()
print(f"\nBirleştirilmiş model ile LoRA'lı model çıktı farkı: {fark:.2e}  → çıkarımda ek maliyet yok")

### Gerçek ölçekte LoRA

LLaMA-2-7B'nin 32 katmanında yalnızca $W_Q$ ve $W_V$ matrislerine ($4096 \times 4096$) LoRA uyguladığımızı düşünelim:

In [ ]:
d, L = 4096, 32
toplam_parametre = 6.74e9
for r in [4, 8, 16, 64]:
    lora_param = L * 2 * r * (d + d)            # her katmanda Q ve V, her biri A(r×d) + B(d×r)
    print(f"r = {r:3d}: {lora_param / 1e6:6.2f} M parametre = modelin %{lora_param / toplam_parametre * 100:.3f}'ü "
          f"| adaptör dosyası ≈ {lora_param * 2 / 2**20:.0f} MB (FP16)")

### Hugging Face PEFT ile LoRA (gerçek bir LLM'de)

```python
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
ayar = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],   # hangi Linear katmanlara takılacak
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, ayar)
model.print_trainable_parameters()
# trainable params: ~1.1M || all params: ~495M || trainable%: ~0.22
```

**QLoRA** (Dettmers vd., 2023): Temel model 4-bit (NF4) kuantize edilir, LoRA adaptörleri 16-bit eğitilir. 65B'lik bir model tek bir 48 GB GPU'da ince ayarlanabilir. Kuantizasyon detayları için → `08_kuantizasyon.ipynb`

---
## Özet

| | KV Cache | LoRA |
|---|---|---|
| Aşama | Çıkarım (inference) | İnce ayar (fine-tuning) |
| Problem | Her adımda geçmişi baştan hesaplamak, $O(n^2)$ | Milyarlarca parametreyi güncellemek: bellek, optimizer durumu, depolama |
| Çözüm | Geçmiş K, V'yi sakla; sadece yeni tokenı işle | $\Delta W = BA$, düşük ranklı güncelleme; $W_0$ donuk |
| Kazanç | Üretim $O(n)$, büyük hızlanma | Eğitilen parametre %0.1–1, görev başına MB'lık adaptör |
| Bedel | Bellek: bağlam × katman × KV kafası | Bazen tam ince ayardan biraz düşük kalite |
| İlgili teknikler | GQA, MQA, PagedAttention (vLLM), KV kuantizasyonu | QLoRA, DoRA, adaptör birleştirme |